# Project: TorchTune

Project Source: [DataCamp - Fine Tuning with Llama 3](https://campus.datacamp.com/courses/fine-tuning-with-llama-3)

In [ ]:
!pip install -qU \
    torchtune \
    datasets \
    bitsandbytes \
    accelerate \
    sentencepiece

In [ ]:
import torch
print(torch.cuda.is_available())

## Step 1: Load dataset

In [1]:
from datasets import load_dataset, Dataset
import yaml
import pprint
import subprocess

In [4]:
ds = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train"
)

In [5]:
print("Columns:", ds.column_names)

Columns: ['flags', 'instruction', 'category', 'intent', 'response']


In [6]:
pprint.pprint(ds[0])

{'category': 'ORDER',
 'flags': 'B',
 'instruction': 'question about cancelling order {{Order Number}}',
 'intent': 'cancel_order',
 'response': "I've understood you have a question regarding canceling order "
             "{{Order Number}}, and I'm here to provide you with the "
             'information you need. Please go ahead and ask your question, and '
             "I'll do my best to assist you."}


## Step 2: (Optional) reduce dataset size for practice/testing

In [7]:
first_thousand_points = ds[:1000]
ds = Dataset.from_dict(first_thousand_points)

print("Dataset shape:", ds.shape)

Dataset shape: (1000, 5)


## Step 3: Preprocess into a single training text field

In [9]:
def merge_example(row):
    row["conversation"] = f"Query: {row['instruction']}\nResponse: {row['response']}"
    return row

ds = ds.map(merge_example)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
print("\nSample conversation:")
print(ds[0]["conversation"])


Sample conversation:
Query: question about cancelling order {{Order Number}}
Response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


## Step 4: Save preprocessed dataset

In [12]:
dataset_path = "preprocessed_dataset"
ds.save_to_disk(dataset_path)

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
print(f"\nSaved dataset to: {dataset_path}")


Saved dataset to: preprocessed_dataset


## Step 5: Create a TorchTune recipe config

In [14]:
config_dict = {
    "batch_size": 4,
    "device": "cuda",          # change to "cpu" if no GPU
    "epochs": 3,
    "output_dir": "/tmp/full-llama3-finetune",
    "model": {
        "_component_": "torchtune.models.llama3_2.llama3_2_1b"
    },
    "optimizer": {
        "_component_": "bitsandbytes.optim.PagedAdamW8bit",
        "lr": 2.0e-05
    },
    "dataset": {
        "_component_": "torchtune.datasets.alpaca_dataset",
        "source": dataset_path
    }
}


In [15]:
yaml_file_path = "custom_recipe.yaml"
with open(yaml_file_path, "w") as yaml_file:
    yaml.dump(config_dict, yaml_file)

In [16]:
print(f"Saved TorchTune config to: {yaml_file_path}")

Saved TorchTune config to: custom_recipe.yaml


## Step 6: Example TorchTune commands

In [17]:
print("\nExample TorchTune commands:")
print("List recipes:")
print("  tune ls")
print("\nRun fine-tuning with a built-in config:")
print("  tune run full_finetune_single_device --config llama3_1/8B_lora_single_device")
print("\nRun fine-tuning with your custom recipe:")
print(f"  tune run --config {yaml_file_path}")


Example TorchTune commands:
List recipes:
  tune ls

Run fine-tuning with a built-in config:
  tune run full_finetune_single_device --config llama3_1/8B_lora_single_device

Run fine-tuning with your custom recipe:
  tune run --config custom_recipe.yaml


## Step 7: Run TorchTune

Mental Model
  - Recipe = training engine
  - Config = settings

In [20]:
!tune ls

RECIPE                                   CONFIG                                  
full_finetune_single_device              llama2/7B_full_low_memory               
                                         code_llama2/7B_full_low_memory          
                                         llama3/8B_full_single_device            
                                         llama3_1/8B_full_single_device          
                                         llama3_2/1B_full_single_device          
                                         llama3_2/3B_full_single_device          
                                         mistral/7B_full_low_memory              
                                         phi3/mini_full_low_memory               
                                         phi4/14B_full_low_memory                
                                         qwen2/7B_full_single_device             
                                         qwen2/0.5B_full_single_device           
                

In [28]:
!tune cat llama3_2/1B_lora_single_device > custom_recipe.yaml

In [29]:
!tune run lora_finetune_single_device --config custom_recipe.yaml dataset=preprocessed_dataset dataset.split=train

INFO:torchtune.utils._logging:Running LoRAFinetuneRecipeSingleDevice with resolved config:

batch_size: 4
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: /tmp/Llama-3.2-1B-Instruct/
  checkpoint_files:
  - model.safetensors
  model_type: LLAMA3_2
  output_dir: /tmp/torchtune/llama3_2_1B/lora_single_device
  recipe_checkpoint: null
clip_grad_norm: null
compile: false
dataset:
  _component_: preprocessed_dataset
  packed: false
  split: train
device: cuda
dtype: bf16
enable_activation_checkpointing: false
enable_activation_offloading: false
epochs: 1
gradient_accumulation_steps: 8
log_every_n_steps: 1
log_peak_memory_stats: true
loss:
  _component_: torchtune.modules.loss.CEWithChunkedOutputLoss
lr_scheduler:
  _component_: torchtune.training.lr_schedulers.get_cosine_schedule_with_warmup
  num_warmup_steps: 100
max_steps_per_epoch: null
metric_logger:
  _component_: torchtune.training.metric_logging.DiskLogger
  log_dir: /tmp/torchtune/llama3_2_1

In [31]:
!tune download meta-llama/Llama-3.2-1B-Instruct

Ignoring files matching the following patterns: None
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
tune download: error: It looks like you are trying to access a gated repository. Please ensure you have access to the repository and have provided the proper Hugging Face API token using the option `--hf-token` or by running `huggingface-cli login`.You can find your token by visiting https://huggingface.co/settings/tokens


In [30]:
# tune run <recipe> --config <config>
!tune run lora_finetune_single_device --config custom_recipe.yaml

INFO:torchtune.utils._logging:Running LoRAFinetuneRecipeSingleDevice with resolved config:

batch_size: 4
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: /tmp/Llama-3.2-1B-Instruct/
  checkpoint_files:
  - model.safetensors
  model_type: LLAMA3_2
  output_dir: /tmp/torchtune/llama3_2_1B/lora_single_device
  recipe_checkpoint: null
clip_grad_norm: null
compile: false
dataset:
  _component_: torchtune.datasets.alpaca_cleaned_dataset
  packed: false
device: cuda
dtype: bf16
enable_activation_checkpointing: false
enable_activation_offloading: false
epochs: 1
gradient_accumulation_steps: 8
log_every_n_steps: 1
log_peak_memory_stats: true
loss:
  _component_: torchtune.modules.loss.CEWithChunkedOutputLoss
lr_scheduler:
  _component_: torchtune.training.lr_schedulers.get_cosine_schedule_with_warmup
  num_warmup_steps: 100
max_steps_per_epoch: null
metric_logger:
  _component_: torchtune.training.metric_logging.DiskLogger
  log_dir: /tmp/torchtune/llam

In [27]:
!tune run lora_finetune_single_device \
  --config llama3_2/1B_lora_single_device \
  dataset=preprocessed_dataset \
  dataset.split=train \
  device=cuda \
  epochs=3 \
  batch_size=4 \
  output_dir=/content/llama-finetune-output

INFO:torchtune.utils._logging:Running LoRAFinetuneRecipeSingleDevice with resolved config:

batch_size: 4
checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: /tmp/Llama-3.2-1B-Instruct/
  checkpoint_files:
  - model.safetensors
  model_type: LLAMA3_2
  output_dir: /content/llama-finetune-output
  recipe_checkpoint: null
clip_grad_norm: null
compile: false
dataset:
  _component_: preprocessed_dataset
  packed: false
  split: train
device: cuda
dtype: bf16
enable_activation_checkpointing: false
enable_activation_offloading: false
epochs: 3
gradient_accumulation_steps: 8
log_every_n_steps: 1
log_peak_memory_stats: true
loss:
  _component_: torchtune.modules.loss.CEWithChunkedOutputLoss
lr_scheduler:
  _component_: torchtune.training.lr_schedulers.get_cosine_schedule_with_warmup
  num_warmup_steps: 100
max_steps_per_epoch: null
metric_logger:
  _component_: torchtune.training.metric_logging.DiskLogger
  log_dir: /content/llama-finetune-output/logs
mode